In [2]:
import pandas as pd
import numpy as np
import sklearn

In [10]:
df = pd.read_csv("data/train.csv")

print(df.columns)
print(df.shape)
print(df.isna().sum)
print(df.describe)
print(df.dtypes)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')
(891, 12)
<bound method DataFrame.sum of      PassengerId  Survived  Pclass   Name    Sex    Age  SibSp  Parch  Ticket  \
0          False     False   False  False  False  False  False  False   False   
1          False     False   False  False  False  False  False  False   False   
2          False     False   False  False  False  False  False  False   False   
3          False     False   False  False  False  False  False  False   False   
4          False     False   False  False  False  False  False  False   False   
..           ...       ...     ...    ...    ...    ...    ...    ...     ...   
886        False     False   False  False  False  False  False  False   False   
887        False     False   False  False  False  False  False  False   False   
888        False     False   False  False  False   True  False  False   False   

In [11]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [12]:
dir(sklearn)

['calibration',
 'callback',
 'clone',
 'cluster',
 'compose',
 'config_context',
 'covariance',
 'cross_decomposition',
 'datasets',
 'decomposition',
 'discriminant_analysis',
 'dummy',
 'ensemble',
 'exceptions',
 'experimental',
 'externals',
 'feature_extraction',
 'feature_selection',
 'frozen',
 'gaussian_process',
 'get_config',
 'impute',
 'inspection',
 'isotonic',
 'kernel_approximation',
 'kernel_ridge',
 'linear_model',
 'manifold',
 'metrics',
 'mixture',
 'model_selection',
 'multiclass',
 'multioutput',
 'naive_bayes',
 'neighbors',
 'neural_network',
 'pipeline',
 'preprocessing',
 'random_projection',
 'semi_supervised',
 'set_config',
 'show_versions',
 'svm',
 'tree']

we split it into x,y first (no need to select features yet, here we just slice out the y column and split it. we can pick later using the pipeline)

In [20]:
from sklearn.model_selection import train_test_split

y = df["Survived"]
x = df.drop(columns=["Survived"])


x_train, x_val, y_train, y_val = train_test_split(x,y, test_size=0.2, stratify=y, random_state=42)

print(x.shape)
print(y.shape)
print(x_train.shape)
print(x_val.shape)

(891, 11)
(891,)
(712, 11)
(179, 11)


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

['ConfusionMatrixDisplay',
 'DetCurveDisplay',
 'DistanceMetric',
 'PrecisionRecallDisplay',
 'PredictionErrorDisplay',
 'RocCurveDisplay',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '_base',
 '_classification',
 '_dist_metrics',
 '_pairwise_distances_reduction',
 '_pairwise_fast',
 '_plot',
 '_ranking',
 '_regression',
 '_scorer',
 'accuracy_score',
 'adjusted_mutual_info_score',
 'adjusted_rand_score',
 'auc',
 'average_precision_score',
 'balanced_accuracy_score',
 'brier_score_loss',
 'calinski_harabasz_score',
 'check_scoring',
 'class_likelihood_ratios',
 'classification_report',
 'cluster',
 'cohen_kappa_score',
 'completeness_score',
 'confusion_matrix',
 'confusion_matrix_at_thresholds',
 'consensus_score',
 'coverage_error',
 'd2_absolute_error_score',
 'd2_brier_score',
 'd2_log_loss_score',
 'd2_pinball_score',
 'd2_tweedie_score',
 'davies_bouldin_score',
 'dcg_score',
 'det_cu

In [43]:
num_features = ["Age", "Fare", "SibSp", "Parch"]
cat_features = ["Sex", "Embarked", "Pclass"]

# preprocessor for num
preprocessor_num = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

# preprocessor for cat
preprocessor_cat = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore"))

# make column to transform cat/num data separately
preprocessor = ColumnTransformer([
    ("nums", preprocessor_num, num_features),
    ("cat", preprocessor_cat, cat_features)
])

# define models so i can loop through them
models = {
    "log_reg": LogisticRegression(),
    "tree": DecisionTreeClassifier(),
    "random_forest": RandomForestClassifier(),
    "HGB": HistGradientBoostingClassifier(),
    "SVM": SVC(C=1),
    "kNN": KNeighborsClassifier()

}

for name, model in models.items():
    # make a pipeline connecting preporcesor with model
    model = make_pipeline(preprocessor, model)
    model.fit(x_train, y_train)
    val_pred = model.predict(x_val)
    train_pred = model.predict(x_train)

    train_accuracy = accuracy_score(train_pred, y_train)
    val_accuracy = accuracy_score(val_pred, y_val)

    print(f"model: {name}")
    print(f"train accuracy: {train_accuracy}")
    print(f"val accuracy: {val_accuracy}")


model: log_reg
train accuracy: 0.8103932584269663
val accuracy: 0.8044692737430168
model: tree
train accuracy: 0.9831460674157303
val accuracy: 0.8156424581005587
model: random_forest
train accuracy: 0.9831460674157303
val accuracy: 0.8156424581005587
model: HGB
train accuracy: 0.949438202247191
val accuracy: 0.8044692737430168
model: SVM
train accuracy: 0.8426966292134831
val accuracy: 0.8156424581005587
model: kNN
train accuracy: 0.851123595505618
val accuracy: 0.8212290502793296


In [ ]:
# unfinished - but you can use loops to try different parameters

# from sklearn.tree import DecisionTreeClassifier

# depths = [0,1,2,3,4,5,6,7,8,9,10]

# for depth in depths:
    